# SQL DATA ANALYSIS & IN-MEMORY ENGINE VALIDATION

In [1]:
# Step 1: Environment Setup & Imports
import os
import sqlite3
import pandas as pd

BASE_DIR = r"C:\Users\AADHYA SHARMA\Downloads\customer-churn-revenue-intelligence"
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
CLEAN_DATA_PATH = os.path.join(PROCESSED_DIR, "ecommerce_churn_clean.csv")

# Step 2: Load Data & Initialize In-Memory SQL Database Engine
df_clean = pd.read_csv(CLEAN_DATA_PATH)
conn = sqlite3.connect(":memory:")

# Load dataframe directly as an in-memory SQL table
df_clean.to_sql("ecommerce_churn", conn, index=False, if_exists="replace")
print(f"Data successfully loaded into SQL table 'ecommerce_churn' ({len(df_clean)} records).")

# Step 3: Analytical SQL Query 1 - Global Business KPIs
query_kpi = """
SELECT 
    COUNT(CustomerID) AS total_customers,
    SUM(CASE WHEN Churn = 0 THEN 1 ELSE 0 END) AS active_customers,
    SUM(CASE WHEN Churn = 1 THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(AVG(Churn) * 100.0, 2) AS churn_rate_pct,
    ROUND((1.0 - AVG(Churn)) * 100.0, 2) AS retention_rate_pct,
    ROUND(SUM(CashbackAmount), 2) AS total_monetary_value,
    ROUND(SUM(CASE WHEN Churn = 1 THEN CashbackAmount ELSE 0 END), 2) AS revenue_at_churn_risk
FROM ecommerce_churn;
"""
print("\n" + "="*70)
print("SQL QUERY 1: GLOBAL BUSINESS KPIS")
print("="*70)
print(pd.read_sql(query_kpi, conn).to_string(index=False))

# Step 4: Analytical SQL Query 2 - Customer Lifecycle Cohorts via CTE
query_cohort = """
WITH CustomerLifecycle AS (
    SELECT 
        CustomerID,
        Churn,
        CashbackAmount,
        CASE 
            WHEN Tenure <= 3 THEN '01. 0-3 Months (New)'
            WHEN Tenure <= 6 THEN '02. 4-6 Months (Early)'
            WHEN Tenure <= 12 THEN '03. 7-12 Months (Mid)'
            WHEN Tenure <= 24 THEN '04. 13-24 Months (Established)'
            ELSE '05. 25+ Months (Mature)'
        END AS tenure_cohort
    FROM ecommerce_churn
)
SELECT 
    tenure_cohort,
    COUNT(CustomerID) AS cohort_size,
    SUM(CASE WHEN Churn = 1 THEN 1 ELSE 0 END) AS churned_count,
    ROUND(AVG(Churn) * 100.0, 2) AS churn_rate_pct,
    ROUND((1.0 - AVG(Churn)) * 100.0, 2) AS retention_rate_pct,
    ROUND(SUM(CashbackAmount), 2) AS cohort_monetary_pool
FROM CustomerLifecycle
GROUP BY tenure_cohort
ORDER BY tenure_cohort;
"""
print("\n" + "="*85)
print("SQL QUERY 2: COHORT RETENTION BREAKDOWN (USING CTE)")
print("="*85)
print(pd.read_sql(query_cohort, conn).to_string(index=False))

# Step 5: Analytical SQL Query 3 - High-Value Churned Customers via Window Functions
query_window = """
WITH CustomerRankings AS (
    SELECT 
        CustomerID,
        PreferedOrderCat,
        Tenure,
        OrderCount,
        CashbackAmount,
        SatisfactionScore,
        Complain,
        Churn,
        DENSE_RANK() OVER (PARTITION BY PreferedOrderCat ORDER BY CashbackAmount DESC) as spending_rank_in_cat
    FROM ecommerce_churn
    WHERE Churn = 1 AND Complain = 1
)
SELECT 
    CustomerID,
    PreferedOrderCat,
    Tenure,
    OrderCount,
    CashbackAmount,
    SatisfactionScore,
    spending_rank_in_cat
FROM CustomerRankings
WHERE spending_rank_in_cat <= 3
ORDER BY PreferedOrderCat, spending_rank_in_cat;
"""
print("\n" + "="*95)
print("SQL QUERY 3: TOP-SPENDING CHURNED CUSTOMERS PER CATEGORY (USING DENSE_RANK)")
print("="*95)
print(pd.read_sql(query_window, conn).to_string(index=False))

# Close connection
conn.close()
print("\nSQL database validation successfully executed.")

Data successfully loaded into SQL table 'ecommerce_churn' (5630 records).

SQL QUERY 1: GLOBAL BUSINESS KPIS
 total_customers  active_customers  churned_customers  churn_rate_pct  retention_rate_pct  total_monetary_value  revenue_at_churn_risk
            5630              4682                948           16.84               83.16             997765.66              152031.64

SQL QUERY 2: COHORT RETENTION BREAKDOWN (USING CTE)
                 tenure_cohort  cohort_size  churned_count  churn_rate_pct  retention_rate_pct  cohort_monetary_pool
          01. 0-3 Months (New)         1560            653           41.86               58.14             242836.21
        02. 4-6 Months (Early)          590             44            7.46               92.54             100244.54
         03. 7-12 Months (Mid)         1584            156            9.85               90.15             258689.28
04. 13-24 Months (Established)         1467             95            6.48               93.52      